In [1]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import os

model = ChatOpenAI(
    model='deepseek-ai/DeepSeek-V3.2',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY')
)


def stream_chat(resp_stream):
    result = ""
    for chunk in resp_stream:
        # 一个chunk就是一个token
        print(chunk.content)
        result += chunk.content

In [2]:
from langchain_community.utilities import SQLDatabase

hostname = 'localhost'
port = '3306'
database = 'pai_coding'
username = 'root'
password = '123456'
mysql_uri = f'mysql+pymysql://{username}:{password}@{hostname}:{port}/{database}?charset=utf8mb4'
db = SQLDatabase.from_uri(mysql_uri);

In [3]:
db.get_usable_table_names()

['DATABASECHANGELOG',
 'DATABASECHANGELOGLOCK',
 'article',
 'article_detail',
 'article_pay_record',
 'article_tag',
 'banner',
 'category',
 'column_article',
 'column_info',
 'comment',
 'config',
 'dict_common',
 'global_conf',
 'notify_msg',
 'read_count',
 'request_count',
 'short_link',
 'short_link_record',
 'tag',
 'user',
 'user_ai',
 'user_ai_history',
 'user_foot',
 'user_info',
 'user_relation']

In [4]:
from langchain_classic.chains.sql_database.query import create_sql_query_chain

# 构造sql chain
sql_chain = create_sql_query_chain(model, db)

In [15]:
resp = sql_chain.invoke(
    {
        "question": "请问表中有多少篇文章",
    }
)
resp

'```sql\nSELECT COUNT(*) AS article_count FROM `article` WHERE `deleted` = 0;\n```'

In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_community.tools import QuerySQLDataBaseTool

# 整合
# 调整提示词
answer_prompt = PromptTemplate.from_template(
    """给定以下用户问题、SQL语句和SQL执行后的结果，回答用户问题。
    Question: {question}
    SQL Query: {query}
    SQL Result: {result}
    回答: """
)
# 创建一个执行sql的tool
execute_sql = QuerySQLDataBaseTool(db=db)

C:\Users\14820\AppData\Local\Temp\ipykernel_26768\3365231013.py:14: LangChainDeprecationWarning: The class `QuerySQLDataBaseTool` was deprecated in LangChain 0.3.12 and will be removed in 1.0. An updated version of the class exists in the `langchain-community package and should be used instead. To use it run `pip install -U `langchain-community` and import as `from `langchain_community.tools import QuerySQLDatabaseTool``.
  execute_sql = QuerySQLDataBaseTool(db=db)


In [19]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

# 1. 生成sql -> 执行sql -> 获取结果 -> 构造回答
def get_sql(text):
    return text.replace('```sql\n', '').replace('\n```', '').strip()


chain = (RunnablePassthrough
         .assign(query=sql_chain | get_sql)  # 生成sql
         .assign(result=itemgetter('query') | execute_sql)  # 获取sql并且执行
         )
chain = chain | answer_prompt | model | StrOutputParser()

In [20]:
resp = chain.invoke({
    "question": "请问表中有多少篇文章",
})

In [21]:
resp

'根据查询结果，表中总共有 8 篇文章。'

部分模型在生成sql的时候带有SQLQuery: 前缀所以会报错

TypeError: unsupported operand type(s) for |: 'function' and 'function'